Steiner systems
===============

**Author:** Rafael



## Sintaxis



Funciones internas pueden "ver" variables definidas exteriormente:



In [1]:
def outer():
    target = 10
    
    def inner():
        # Python doesn't find 'target' inside inner(), 
        # so it looks out to outer() and finds it.
        print(target) 
        
    inner()

outer()

Pero no pueden modificarlas:



In [1]:
def outer():
    num_blocks = 0
    
    def inner():
        num_blocks += 1  
        
    inner()

# outer()

Tenemos que declarar la variable como `nonlocal`.



In [1]:
def outer():
    num_blocks = 0
    
    def inner():
        nonlocal num_blocks
        num_blocks += 1
        print(num_blocks)
        
    inner()

outer()

Si la variable es mutable no se necesita nonlocal:



In [1]:
def outer():
    blocks = []
    
    def inner():
        blocks.append("hola")
        print(blocks)
        
    inner()

outer()

## Steiner with hill climbing



In [1]:
import random

def steiner_system(v):
    """
    Constructs an STS(v) using the core logic of Stinson's hill-climbing,
    """
    if v % 6 not in [1, 3]:
        raise ValueError("STS only exists for v ≡ 1 or 3 (mod 6)")

    # The Adjacency Matrix: third_point[x][y] = z
    third_point = [[0] * (v + 1) for _ in range(v + 1)]
    
    # The Active Set (L): Points that still need pairs
    live_points = list(range(1, v + 1))
    
    # The Shadow Graph: Dictionary mapping a point to a simple list of its available neighbors
    uncovered_pairs = {x: [y for y in range(1, v + 1) if y != x] for x in range(1, v + 1)}
    
    num_blocks = 0
    target_blocks = v * (v - 1) // 6

    def reactivate_pair(x, y):
        """Adds edge (x, y) back into the shadow graph."""
        # If x had no available pairs, it wasn't in the active set. Wake it up!
        if len(uncovered_pairs[x]) == 0:
            live_points.append(x)
            
        uncovered_pairs[x].append(y)

    def deactivate_pair(x, y):
        """Removes edge (x, y) from the shadow graph because it's now in a block."""
        uncovered_pairs[x].remove(y)
        
        # If x has run out of available pairs, it is saturated. Remove it from the active set.
        if len(uncovered_pairs[x]) == 0:
            live_points.remove(x)

    def add_triple(x, y, z):
        nonlocal num_blocks
        # 1. Record the block in the adjacency matrix
        third_point[x][y] = z; third_point[y][x] = z
        third_point[x][z] = y; third_point[z][x] = y
        third_point[y][z] = x; third_point[z][y] = x
        
        # 2. Consume the 3 edges (6 directed pairs) from the shadow graph
        for a, b in [(x, y), (y, x), (x, z), (z, x), (y, z), (z, y)]:
            deactivate_pair(a, b)
            
        num_blocks += 1

    def exchange_triple(x, y, z, w):
        # 1. Erase the old block {w, y, z} from the adjacency matrix
        third_point[w][y] = 0; third_point[y][w] = 0
        third_point[w][z] = 0; third_point[z][w] = 0
        third_point[y][z] = 0; third_point[z][y] = 0
        
        # 2. Return those 3 edges back to the shadow graph
        for a, b in [(w, y), (y, w), (w, z), (z, w), (y, z), (z, y)]:
            reactivate_pair(a, b)
            
        nonlocal num_blocks
        num_blocks -= 1
        
        # 3. Add the new block {x, y, z}
        add_triple(x, y, z)

    # --- Main Hill-Climbing Loop ---
    while num_blocks < target_blocks:
        # Step 1: Pick an active point x
        x = random.choice(live_points)
        
        # Step 2: Pick two available neighbors for x
        y, z = random.sample(uncovered_pairs[x], 2)
        
        # Step 3: Check if edge (y, z) is free
        w = third_point[y][z]
        if w == 0:
            add_triple(x, y, z)
        else:
            exchange_triple(x, y, z, w)

    # --- Extraction ---
    triples = []
    for i in range(1, v + 1):
        for j in range(i + 1, v + 1):
            k = third_point[i][j]
            if k > j:
                triples.append([i, j, k])
                
    return triples

In [1]:
sts=steiner_system(1201)

In [1]:
len(sts)

## Optimization



In [1]:
import timeit
import random

def test_deletions():
    N = 50_000               # Size of our list
    deletions = 5_000        # How many items we want to delete
    
    # We will delete the same random targets in both tests
    targets = random.sample(range(N), deletions)

    # ---------------------------------------------------------
    # 1. The Standard Way: list.remove()
    # ---------------------------------------------------------
    def standard_way():
        my_list = list(range(N))
        
        for target in targets:
            my_list.remove(target) # O(N) search and shift
            
        return my_list

    # ---------------------------------------------------------
    # 2. The Optimized Way: Dictionary Map + Swap & Pop
    # ---------------------------------------------------------
    def optimized_way():
        my_list = list(range(N))
        
        # Build the index map: {value: current_index}
        position = {val: idx for idx, val in enumerate(my_list)}
        
        for target in targets:
            # 1. Find the target instantly
            idx_to_remove = position[target]
            
            # 2. Find the last element
            last_element = my_list[-1]
            
            # 3. Overwrite the target with the last element
            my_list[idx_to_remove] = last_element
            
            # 4. Update the dictionary so it knows the last element moved!
            position[last_element] = idx_to_remove
            
            # 5. Remove the last slot and clean up the dictionary
            my_list.pop()
            del position[target]
            
        return my_list

    # --- Verification & Benchmarking ---
    
    # Prove formal equivalence (Sets are used to ignore the scrambled order)
    result_standard = standard_way()
    result_optimized = optimized_way()
    print(f"Formal Equivalence (Same elements left?): {set(result_standard) == set(result_optimized)}")
    
    # Measure Time
    t1 = timeit.timeit(standard_way, number=1)
    t2 = timeit.timeit(optimized_way, number=1)
    
    print(f"Standard remove() took: {t1:.4f} seconds")
    print(f"Swap & Pop took:        {t2:.4f} seconds")
    print(f"Speedup:                {t1/t2:.1f}x faster")

test_deletions()

## Steiner with optimization



In [1]:
import random

def steiner_triple_system(v):
    """
    Constructs a Steiner triple system on v points using the revised
    Stinson hill-climbing algorithm (Algorithm 5.19).
    """
    if v % 6 not in [1, 3]:
        raise ValueError("STS only exists for v ≡ 1 or 3 (mod 6)")

    # third_point[x][y] stores z if {x, y, z} is a triple, else 0.
    # Using v+1 to allow 1-based indexing for points.
    third_point = [[0] * (v + 1) for _ in range(v + 1)]
    
    # Points with at least one uncovered pair
    live_points = list(range(1, v + 1))
    # point_position is a dictionary
    point_position = {x: i for i, x in enumerate(live_points)}
    num_live_points = v
    
    # Tracking uncovered pairs for each point
    num_uncovered_pairs = {x: v - 1 for x in range(1, v + 1)}
    uncovered_pairs = {x: [y for y in range(1, v + 1) if y != x] for x in range(1, v + 1)}
    pair_position = {x: {y: i for i, y in enumerate(uncovered_pairs[x])} for x in range(1, v + 1)}
    
    num_blocks = 0
    target_blocks = v * (v - 1) // 6

    def reactivate_pair(x, y):
        nonlocal num_live_points
        # If point was inactive (no uncovered pairs), bring it back to live_points
        if num_uncovered_pairs[x] == 0:
            live_points[num_live_points] = x
            point_position[x] = num_live_points
            num_live_points += 1
        
        pos = num_uncovered_pairs[x]
        uncovered_pairs[x][pos] = y
        pair_position[x][y] = pos
        num_uncovered_pairs[x] += 1

    def deactivate_pair(x, y):
        nonlocal num_live_points
        pos = pair_position[x][y]
        last_neighbor_idx = num_uncovered_pairs[x] - 1
        
        # Swap last uncovered neighbor into the current pair's position
        displaced = uncovered_pairs[x][last_neighbor_idx]
        uncovered_pairs[x][pos] = displaced
        pair_position[x][displaced] = pos
        
        num_uncovered_pairs[x] -= 1
        
        # If point has no more uncovered pairs, remove from live_points
        if num_uncovered_pairs[x] == 0:
            pos_in_live = point_position[x]
            last_point = live_points[num_live_points - 1]
            live_points[pos_in_live] = last_point
            point_position[last_point] = pos_in_live
            num_live_points -= 1

    def add_triple(x, y, z):
        nonlocal num_blocks
        third_point[x][y] = z; third_point[y][x] = z
        third_point[x][z] = y; third_point[z][x] = y
        third_point[y][z] = x; third_point[z][y] = x
        
        # Deactivate all 6 directed pairs involved in the triple
        for a, b in [(x, y), (y, x), (x, z), (z, x), (y, z), (z, y)]:
            deactivate_pair(a, b)
        num_blocks += 1

    def exchange_triple(x, y, z, w):
        # Remove existing triple {w, y, z}
        third_point[w][y] = 0; third_point[y][w] = 0
        third_point[w][z] = 0; third_point[z][w] = 0
        third_point[y][z] = 0; third_point[z][y] = 0
        
        # Reactivate pairs from the removed triple
        for a, b in [(w, y), (y, w), (w, z), (z, w), (y, z), (z, y)]:
            reactivate_pair(a, b)
            
        # Add the new triple {x, y, z} (num_blocks stays the same net total)
        # We manually decrement so add_triple's increment results in no net change
        nonlocal num_blocks
        num_blocks -= 1
        add_triple(x, y, z)

    # Main Hill-Climbing Loop
    while num_blocks < target_blocks:
        # Pick random point x from those that have uncovered pairs
        idx_x = random.randint(0, num_live_points - 1)
        x = live_points[idx_x]
        
        # Pick two random distinct neighbors y, z that are currently "uncovered" for x
        # num_uncovered_pairs[x] is at least 2 because a triple requires 2 neighbors
        indices = random.sample(range(num_uncovered_pairs[x]), 2)
        y = uncovered_pairs[x][indices[0]]
        z = uncovered_pairs[x][indices[1]]
        
        w = third_point[y][z]
        if w == 0:
            # If y and z are also free, we just add the block
            add_triple(x, y, z)
        else:
            # If {y, z} is already in triple {w, y, z}, perform an exchange (the "switch")
            exchange_triple(x, y, z, w)

    # Extraction: Convert the matrix into a list of unique triples
    triples = []
    for i in range(1, v + 1):
        for j in range(i + 1, v + 1):
            k = third_point[i][j]
            # Only record if i < j < k to avoid permutations
            if k > j:
                triples.append([i, j, k])
                
    return triples

In [1]:
steiner_triple_system(9)